## Week 1 - Lab 4: Your Digital Twin (Part 2) — Tool Use

Today we level up the personal chatbot with **tool use** and then **deploy it**.

New capabilities:
- The bot can capture visitor email addresses and send them to your phone
- The bot logs questions it can't answer so you can review them later
- We deploy the finished product to HuggingFace Spaces

This is the **"Professionally You"** project — a real, deployable AI agent.

## Step 0: Set up Pushover (push notifications)

Pushover sends notifications to your phone when someone interacts with your chatbot.

1. Sign up at [https://pushover.net](https://pushover.net)
2. Create an application called "Career Bot" (or any name)
3. Add to your `.env` file:
   ```
   PUSHOVER_USER=your_user_key
   PUSHOVER_TOKEN=your_app_token
   ```
4. Install the Pushover app on your phone

**This is optional** — if you skip it, just comment out the `push()` calls.

In [ ]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader
import gradio as gr
import requests
import json
import os

load_dotenv(override=True)
client = genai.Client()

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(f"Pushover user: {'✓' if pushover_user else '✗ not set (optional)'}")
print(f"Pushover token: {'✓' if pushover_token else '✗ not set (optional)'}")

In [ ]:
def push(message: str):
    """Send a push notification via Pushover."""
    if not pushover_user or not pushover_token:
        print(f"[Push skipped — no Pushover keys] {message}")
        return
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    print(f"[Push sent] {message}")

In [ ]:
# Test it!
push("Career Bot is online! 🚀")

## Define the tools

With google-genai, you define tools as **plain Python functions**.
The SDK reads your type hints and docstring to tell Gemini what the tool does.
No JSON schema needed — much cleaner than the OpenAI approach!

Gemini will call these functions when it decides they're needed.

In [ ]:
def record_user_details(email: str, name: str = "Not provided", notes: str = "Not provided") -> dict:
    """Record that a visitor provided their email and is interested in contact.

    Args:
        email: The visitor's email address
        name: The visitor's name, if they provided it
        notes: Any context about their interests worth recording
    """
    push(f"New contact: {name} — {email} — {notes}")
    return {"recorded": "ok"}


def record_unknown_question(question: str) -> dict:
    """Record a question that couldn't be answered so the owner can review it.

    Use this whenever you don't know the answer, even for trivial questions.

    Args:
        question: The question that couldn't be answered
    """
    push(f"Unanswered question: {question}")
    return {"recorded": "ok"}

In [ ]:
# *** CHANGE THIS ***
name = "Your Name"

In [ ]:
# Load profile data

linkedin = ""
if os.path.exists("me/linkedin.pdf"):
    reader = PdfReader("me/linkedin.pdf")
    for page in reader.pages:
        text = page.extract_text()
        if text:
            linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
system_prompt = (
    f"You are acting as {name} on their personal website. "
    f"Answer questions about {name}'s career, background, skills and experience. "
    f"Be professional and engaging — as if talking to a potential employer or client. "
    f"If you don't know the answer to any question, ALWAYS use your record_unknown_question tool to log it. "
    f"When a visitor shows interest, ask for their email and record it using record_user_details.\n\n"
    f"## Summary:\n{summary}\n\n"
)
if linkedin:
    system_prompt += f"## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"Always stay in character as {name}."

In [ ]:
def history_to_contents(history: list[dict]) -> list:
    contents = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        contents.append(
            types.Content(role=role, parts=[types.Part(text=msg["content"])])
        )
    return contents

In [ ]:
def handle_tool_calls(response) -> list:
    """Execute all tool calls from a model response and return result parts."""
    result_parts = []
    for fc in response.function_calls:
        print(f"Tool called: {fc.name}")
        tool_fn = globals().get(fc.name)
        result = tool_fn(**dict(fc.args)) if tool_fn else {"error": "unknown tool"}
        result_parts.append(
            types.Part.from_function_response(
                name=fc.name,
                response={"result": str(result)}
            )
        )
    return result_parts

In [ ]:
config = types.GenerateContentConfig(
    system_instruction=system_prompt,
    tools=[record_user_details, record_unknown_question]
)


def chat(message: str, history: list[dict]) -> str:
    contents = history_to_contents(history)
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    # Agentic loop — keep going until no more tool calls
    while True:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=contents,
            config=config
        )

        if not response.function_calls:
            # No tool calls — we're done
            break

        # Add model response to history
        contents.append(response.candidates[0].content)

        # Execute tools and add results to history
        tool_parts = handle_tool_calls(response)
        contents.append(types.Content(role="user", parts=tool_parts))

    return response.text

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

## Test it!

Try asking:
- Questions about your background (should answer from the profile)
- A question it can't answer (should trigger `record_unknown_question`)
- Express interest in working together (should ask for your email, then trigger `record_user_details`)

Watch the terminal — you'll see tool calls printed as they happen.

## Deploying to HuggingFace Spaces

The code in `app.py` packages this chatbot for deployment.

### Steps:

1. **Create a HuggingFace account** at [https://huggingface.co](https://huggingface.co)
2. **Generate a write token**: Avatar → Access Tokens → New Token (Write permission)
3. **Install the HF CLI and login:**
   ```bash
   uv tool install 'huggingface_hub[cli]'
   hf auth login --token YOUR_TOKEN_HERE
   hf auth whoami  # verify you're logged in
   ```
4. **From the `1_foundations/` folder, deploy:**
   ```bash
   cd 1_foundations
   uv run gradio deploy
   ```
5. Follow the prompts: name it `career-bot`, choose `app.py`, select `cpu-basic`, and enter your secrets (`GEMINI_API_KEY`, `PUSHOVER_USER`, `PUSHOVER_TOKEN`)

### Important before deploying:
- Update `app.py` — change `self.name = "Your Name"` to your actual name
- Make sure `me/summary.txt` has your real profile
- Add `me/linkedin.pdf` if you want to include your LinkedIn data

### Managing secrets after deployment:
1. Log into HuggingFace → your profile → your Space
2. Settings wheel (top right) → Variables and Secrets

## Exercise

- **Deploy it** for yourself — this is a real portfolio piece!
- **Add more context** to `summary.txt` — better context = better answers
- **Add more tools**: What else could it do?
  - Look up a calendar and book a meeting
  - Query a FAQ database
  - Send an email notification instead of/in addition to Pushover
- **Combine with the evaluator** from Lab 3 — add quality checking before returning replies